# HighPitchCT — Stage 4 على Kaggle (`v0.7.0`): محرك الإسقاط الأمامي (`2D Fan-Beam` + `3D Helical Cone-Beam`) ومحاكاة `RDC` و`ITF` (`AAPM 2026 Fig. 2(a)-(d)`)

1. فعّل **Internet = On** وأضف `GITHUB_TOKEN` في **Add-ons → Secrets** (ويمكنك تفعيل **GPU T4 x2** ليختار الدفتر تلقائيًا `--backend torch_cuda` صراحةً).
2. ينفّذ هذا الدفتر:
   - التحديث إلى `v0.7.0` وتشغيل الـ **`47` اختبارًا كاملةً** مع `-W error::DeprecationWarning`.
   - إعادة تشغيل `Stage 3` بمرشح **`ram_lak`** ومجال رؤية مطابق لصور الشركة (**`FOV = 400.0 mm`**) للتحقق من اختفاء فرق التكبير البصري (`1:1 Millimeter Scale`).
   - تشغيل `hpct stage4-sim --case-id C012 --scenario both --target-pitch 2.4 --pitches 2.4,3.2,4.8` لتوليد أزواج التدريب ثنائية النطاق (`RDC` الشكل `1(b)` + `ITF` الشكل `1(c)` + إعادة الإسقاط $S_{\text{reproj}}$ الشكل `2(d)`).
   - عرض جميع **لوحات المقارنة البصرية لـ `Stage 3` المحدثة و`Stage 4` (`Visual QA PNGs`)** جنبًا إلى جنب.


In [ ]:
import os
from pathlib import Path
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret('GITHUB_TOKEN')
repo = Path('/kaggle/working/HighPitchCT-Pipeline')
if not repo.exists():
    rc = os.system(f'git clone https://{token}@github.com/BrahimSoufghalem/HighPitchCT-Pipeline.git {repo}')
    assert rc == 0, 'فشل استنساخ المستودع؛ تحقق من Internet ومن GITHUB_TOKEN'
else:
    rc = os.system(f'git -C {repo} pull --ff-only')
    assert rc == 0, 'فشل تحديث المستودع عبر git pull'

%cd /kaggle/working/HighPitchCT-Pipeline
!python -m pip install -q -e .
!python -W error::DeprecationWarning -m unittest discover -s tests -t . -v


## 1. تشغيل `Stage 3` المحدّثة (`ram_lak` + `FOV = 400.0 mm`) ثم `Stage 4` (`RDC` + `ITF` + `Fig. 2(a)-(d)`)


In [ ]:
import torch

backend = 'torch_cuda' if torch.cuda.is_available() else 'numpy'
print(f'Explicit backend selected: {backend}')

!hpct download --series-file configs/pilot_c012.csv --workers 4 --retries 5
!hpct stage3-recon --case-id C012 --num-slices 1 --image-size 512 --filter-name ram_lak --backend {backend} --run-phantom-check --save-reference-npz --workers 4
!hpct stage4-sim --case-id C012 --scenario both --target-pitch 2.4 --pitches 2.4,3.2,4.8 --num-slices 1 --n-xy 256 --itf-slab-slices 15 --filter-name ram_lak --backend {backend} --run-phantom-check --save-pairs-npz --workers 4


## 2. ملخص المقاييس الفيزيائية والكمية لـ `Stage 4` (`Forward->FBP Roundtrip` + `RDC` مقابل `ITF` عبر `p = 2.4, 3.2, 4.8`)


In [ ]:
import json
import pandas as pd

with open('/kaggle/working/highpitchct-data/manifests/stage4_sim_report.json', encoding='utf-8') as f:
    rep4 = json.load(f)

assert rep4['overall_pass'] and rep4['simulated_series_count'] == 1, 'فشل تدقيق Stage 4 على C012!'

s4_c012 = rep4['simulated_series'][0]
s4_phan = rep4['synthetic_phantom_forward_check']

rows = [
    {
        'الاختبار / السيناريو': 'Phantom 2D Scalar Parity (Vectorized vs Scalar Ref)',
        'FOV / Pitch': '360.0 mm',
        'RMSE [HU]': '-',
        'SSIM': '-',
        'Missing-Wedge MAE (Raw -> Reproj)': f"max diff = {s4_phan['scalar_parity_2d_forward']['max_abs_line_integral_diff']:.2e}",
    },
    {
        'الاختبار / السيناريو': 'Phantom 2D Forward -> Ram-Lak FBP Roundtrip',
        'FOV / Pitch': '360.0 mm / p=0.85',
        'RMSE [HU]': f"{s4_phan['forward_to_fbp_roundtrip_vs_gt']['rmse_hu']:.2f}",
        'SSIM': f"{s4_phan['forward_to_fbp_roundtrip_vs_gt']['ssim']:.4f}",
        'Missing-Wedge MAE (Raw -> Reproj)': '100% Measured',
    },
]

for p_key, p_sum in s4_c012['rdc_multi_pitch_summary'].items():
    sm = p_sum['sinogram_metrics']
    pm = p_sum['prelim_vs_target_metrics']
    rows.append({
        'الاختبار / السيناريو': f"C012 Scenario 1: RDC ({p_key}, rows={p_sum['active_row_range']})",
        'FOV / Pitch': f"{p_sum['fov_mm']:.1f} mm / eff.p={p_sum['effective_pitch']:.2f}",
        'RMSE [HU]': f"{pm['rmse_hu']:.2f} (raw: {p_sum['raw_nofill_vs_target_metrics']['rmse_hu']:.2f})",
        'SSIM': f"{pm['ssim']:.4f}",
        'Missing-Wedge MAE (Raw -> Reproj)': f"{sm['raw_missing_wedge_mae']:.4f} -> {sm['reproj_missing_wedge_mae']:.4f}",
    })

if s4_c012['primary_itf_summary'] is not None:
    itf_s = s4_c012['primary_itf_summary']
    sm_i = itf_s['sinogram_metrics']
    pm_i = itf_s['prelim_vs_target_metrics']
    rows.append({
        'الاختبار / السيناريو': f"C012 Scenario 2: ITF 3D Helical Forward (64 rows)",
        'FOV / Pitch': f"{itf_s['fov_mm']:.1f} mm / eff.p={itf_s['effective_pitch']:.2f}",
        'RMSE [HU]': f"{pm_i['rmse_hu']:.2f} (raw: {itf_s['raw_nofill_vs_target_metrics']['rmse_hu']:.2f})",
        'SSIM': f"{pm_i['ssim']:.4f}",
        'Missing-Wedge MAE (Raw -> Reproj)': f"{sm_i['raw_missing_wedge_mae']:.4f} -> {sm_i['reproj_missing_wedge_mae']:.4f}",
    })

display(pd.DataFrame(rows))


## 3. المقارنة البصرية الإلزامية (`Stage 3 Updated FOV=400mm` + `Stage 4 Phantom & Real C012 RDC/ITF Visual QA`)


In [ ]:
from IPython.display import Image, Markdown, display

with open('/kaggle/working/highpitchct-data/manifests/stage3_recon_report.json', encoding='utf-8') as f:
    rep3 = json.load(f)

figs = [
    ('Stage 3 (محدّث v0.7.0) — المريض C012 بمرشح ram_lak ومطابقة FOV = 400.0 mm (1:1 مع صورة Siemens)', rep3['reconstructed_series'][0]['visual_qa_png']),
    ('Stage 4 — فانتوم الصدر ثلاثي الأبعاد: إثبات Roundtrip (Forward -> Ram-Lak FBP) ومحاكاة RDC و ITF', s4_phan['visual_qa_png']),
    ('Stage 4 — المريض الحقيقي C012: الشكل 2(a)-(d) الكامل (الساينوغرام المفقود -> المعاد إسقاطه s_reproj + مقارنة RDC مقابل ITF)', s4_c012['visual_qa_png']),
]

for title, png_path in figs:
    display(Markdown(f'### {title}'))
    display(Image(filename=png_path))
